In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import gseapy
import seaborn as sns
import json

matplotlib.rcParams['pdf.fonttype'] = 42


In [ ]:
fdr_threshold = 0.2

In [ ]:
cell_types = ['HSC', 'monocyte']

combined_res = {}
for cell_type in cell_types:
    DE = pd.read_csv('../output/' + cell_type + '_jak2_de_individual_patients.csv', index_col = 0)
    DE['names'] = DE.index

    combined_res[cell_type] = pd.DataFrame()
    for patient in DE.columns:
        if patient in ['names', 'scores']:
            continue
        
        print(patient)
        DE['scores'] = DE[patient]

        pre_res = gseapy.prerank(DE.loc[:,['names', 'scores']].dropna(), gene_sets='MSigDB_Hallmark_2020', verbose=True)
        filtered_res = pre_res.res2d[pre_res.res2d['FDR q-val'] < fdr_threshold].sort_values('NES', ascending=True)
        filtered_res['sample'] = patient

        combined_res[cell_type] = pd.concat([combined_res[cell_type], filtered_res], axis=0)

In [ ]:
### find number of patients with significant enrichment for each term

terms_combined = {}
for cell_type in combined_res.keys():
    print(cell_type)
    terms = combined_res[cell_type].loc[combined_res[cell_type]['ES'] > 0]['Term'].value_counts()
    terms = terms[terms >= 3]
    terms_combined[cell_type] = terms.index
    print('mutation enriched:', terms)
    terms = combined_res[cell_type].loc[combined_res[cell_type]['ES'] < 0]['Term'].value_counts()
    terms = terms[terms >= 3]
    print('wildtype enriched:', terms)
    terms_combined[cell_type] = terms_combined[cell_type].append(terms.index)
